In [8]:
import re
import glob

def extract_usis_from_error_file(filename):
    with open(filename, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()

        # Find all USI blocks starting with 'USI: {number}: '
        usi_blocks = re.split(r'^USI \d+: ', content, flags=re.MULTILINE)[1:]  # Skip first empty element
        
        unvalid_usis = []
        other_usis = []
        
        for block in usi_blocks:
            # Extract the USI identifier (first line of the block)
            lines = block.strip().split('\n')
            if lines:
                usi = lines[0]
                block_content = '\n'.join(lines[1:])  # Rest of the content
                
                # Check if this USI is invalid based on the error message
                if 'Invalid universal spectrum identifier' in block_content:
                    unvalid_usis.append(usi)
                else:
                    other_usis.append(usi)

        return unvalid_usis, other_usis
# Process all error files matching the pattern
for error_file in glob.glob('usis_*_error.txt'):
    # Extract the label for output file
    label = error_file.replace('usis_', '').replace('_error.txt', '').replace('_', '_')
    output_file = f"{label}_error_extraction.txt"

    unvalid_usis, other_usis = extract_usis_from_error_file(error_file)

    # Extract dataset numbers from unvalid_usis
    unvalid_datasets = set()
    for usi in unvalid_usis:
        # Assuming dataset number is the second part after splitting by ':'
        parts = usi.split(':')
        if len(parts) > 1:
            unvalid_datasets.add(parts[1])

    with open(output_file, 'w', encoding='utf-8') as out:
        out.write("Unvalid USIs:\n")
        for usi in unvalid_usis:
            out.write(usi + '\n')
        out.write("\nUnvalid USI dataset numbers:\n")
        for ds in sorted(unvalid_datasets):
            out.write(ds + '\n')
        out.write("\nOther USIs:\n")
        for usi in other_usis:
            out.write(usi + '\n')
        # Extract and write sorted dataset numbers for other_usis
        other_datasets = set()
        for usi in other_usis:
            parts = usi.split(':')
            if len(parts) > 1:
                other_datasets.add(parts[1])
        out.write("\nOther USI dataset numbers:\n")
        for ds in sorted(other_datasets):
            out.write(ds + '\n')

also put the unvalid usi dataset number (a set ) under the reason of unvalidation in the file